# 05 — Modelo de Taxas (Modelo B), refinamento do Modelo A e avaliação

Este notebook lê o que o `04_Preparacao_Modelagem` salvou (`dataset_modelagem.csv`, folds e resultados) e dá os próximos passos **sem depender do GDELT**.

## O que o 04 mostrou e o que este notebook faz com isso

| Achado do 04 | Consequência | O que o 05 faz |
|---|---|---|
| Taxas caíram de forma estrutural (BTC ~23x, ETH ~125x em 2 anos) e o limite de gás do ETH dobrou | Variáveis em **nível** são não estacionárias | Usa só variáveis **relativas** (vs. mediana de 30 dias) e testa se isso corrige a piora do on-chain no Modelo A |
| `ev_x` (≥ 2x a mediana de 7 dias) muda de prevalência ao longo do tempo (efeito de piso) | O evento relativo perde significado econômico | Mede a deriva, usa `ev_q` (quantil móvel) como alvo principal e acrescenta o alvo de **início de evento** |
| Granger nos dois sentidos, sinais inconsistentes; no ETH a taxa já sobe antes do pico de RV | Não há evidência de precedência; há **co-movimento na mesma hora com persistência** | Troca "RV antecede taxa" por "o choque de hoje prevê a taxa das próximas horas, **além do próprio estado da taxa**" |
| Lift de 2 a 3,7x, mas sem controlar a persistência da taxa | O lift pode ser só "taxa alta continua alta" | Escada do Modelo B começa pelo **estado da própria taxa** e mede o ganho incremental do mercado, do fluxo e do Fear & Greed |
| Fear & Greed × taxa: ρ = +0,64 (BTC) e +0,74 (ETH) | Provável **correlação espúria** por tendência comum | Refaz com taxa dessazonalizada e em diferenças |
| Em BTC, 25% das horas têm ocupação < 0,94 | A cauda baixa indica **mempool drenada** (sinal de ausência de fila) | Testa a ocupação como indicador de fila |
| Custos de centavos | "Esperar para economizar" só faz sentido se **houver queda prevista** | Prevê a razão de custo `t+6 / t+1` e avalia a política "esperar só quando a queda prevista é forte" |
| On-chain e F&G pioram o HAR fora da amostra | Sem GDELT, o sentimento textual não pode ser testado | Substitui por proxies de **fluxo de ordens** (compra agressiva, já nos arquivos da Binance) |

## Regras mantidas
1. O *holdout* (últimos 90 dias) **só é aberto na seção final**, uma única vez, com as especificações já escolhidas por regra fixa em `dev`. A célula tem trava.
2. Toda feature nova é causal e passa por **teste de perturbação**.
3. Nenhuma conclusão é escrita à mão: tabelas e veredictos saem de números calculados.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    TEM_SKLEARN = True
except ImportError:
    TEM_SKLEARN = False
    print("scikit-learn não encontrado: o modelo de árvores (gradient boosting) será pulado.")

# ------------------- CAMINHOS (ajuste se necessário) -------------------
PASTA_BASE = Path("/home/alisson/datathon fgv")
PASTA_DADOS = PASTA_BASE / "Análises"
PASTA_2ANOS = PASTA_BASE / "dados_cripto" / "historico_2anos"
PASTA_MODELAGEM = PASTA_DADOS / "modelagem"          # saídas do notebook 04
PASTA_SAIDA = PASTA_MODELAGEM / "05"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ------------------- PARÂMETROS -------------------
HORIZONTES_VOL = [1, 4, 12, 24]
HORIZONTES_TAXA = [1, 6]
SERIES_TAXA = {"btc": "btc_bq_taxa_p50_sat_vb", "eth": "eth_bq_base_fee_media_gwei"}
H_PRAZO = 6                      # prazo máximo de espera na simulação operacional (horas)
TAXAS_ALERTA = [0.05, 0.10, 0.20]
L2_LOGIT = 1.0
B_BOOT = 500
BLOCO_H = 48
SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

# ------------------- HOLDOUT: TRAVA -------------------
ABRIR_HOLDOUT = False            # só mude para True na seção final, com as escolhas já fixadas
FORCAR_REABERTURA = False        # reabrir o holdout invalida o rigor do teste
print("Configuração carregada | sklearn:", TEM_SKLEARN, "| holdout aberto:", ABRIR_HOLDOUT)

## 1. Carga e leitura automática dos resultados do 04

In [ ]:
df_all = pd.read_csv(PASTA_MODELAGEM / "dataset_modelagem.csv")
df_all["hora"] = pd.to_datetime(df_all["hora"], utc=True).astype("datetime64[ns, UTC]")
df_all = df_all.sort_values("hora").reset_index(drop=True)
folds_df = pd.read_csv(PASTA_MODELAGEM / "folds_walkforward.csv")
folds = folds_df[["tr_ini", "tr_fim", "te_ini", "te_fim"]].to_dict("records")
inc04 = pd.read_csv(PASTA_MODELAGEM / "resultado_escada_incremental.csv")

# fluxo de ordens (compra agressiva) a partir do arquivo de 1h da Binance, se existir
ARQ_1H = PASTA_2ANOS / "mercado_1h.csv"
if ARQ_1H.exists():
    m1 = pd.read_csv(ARQ_1H, usecols=["abertura_ts", "ativo", "volume_quote", "volume_taker_quote"])
    m1["hora"] = pd.to_datetime(m1["abertura_ts"], utc=True).astype("datetime64[ns, UTC]")
    m1["taker_ratio"] = m1["volume_taker_quote"] / m1["volume_quote"].where(m1["volume_quote"] > 0)
    piv = m1.pivot_table(index="hora", columns="ativo", values="taker_ratio", aggfunc="mean")
    piv.columns = [f"{c.lower()}_taker_ratio" for c in piv.columns]
    df_all = df_all.merge(piv.reset_index(), on="hora", how="left", validate="one_to_one")
    TEM_FLUXO = True
    print("Fluxo de ordens incorporado:", [c for c in df_all.columns if c.endswith("_taker_ratio")])
else:
    TEM_FLUXO = False
    print("AVISO: mercado_1h.csv não encontrado -> sem proxies de fluxo de ordens.")

print(f"Dataset: {len(df_all):,} linhas | partições:", df_all["particao"].value_counts().to_dict())

# ---------------- leitura automática da escada do 04 ----------------
inc04["sinal"] = np.where(inc04["ganho_QLIKE_%"] > 0, "melhora", "piora")
res = inc04.groupby("degrau").agg(
    n_testes=("p_valor", "size"),
    ganho_medio_pct=("ganho_QLIKE_%", "mean"),
    n_melhora=("ganho_QLIKE_%", lambda s: int((s > 0).sum())),
    n_sig_p05=("p_valor", lambda s: int((s < 0.05).sum())),
    n_sig_Holm=("p_Holm", lambda s: int((s < 0.05).sum())),
)
pior = inc04[(inc04["p_Holm"] < 0.05) & (inc04["ganho_QLIKE_%"] < 0)].groupby("degrau").size()
melhor = inc04[(inc04["p_Holm"] < 0.05) & (inc04["ganho_QLIKE_%"] > 0)].groupby("degrau").size()
res["n_piora_sig_Holm"] = pior.reindex(res.index).fillna(0).astype(int)
res["n_melhora_sig_Holm"] = melhor.reindex(res.index).fillna(0).astype(int)
res = res.drop(columns="n_sig_Holm", errors="ignore")
print("\nResumo automático da escada de volatilidade do 04 (por degrau):")
display(res.round(3))

# regimes: razão primeiro/último trimestre completo
tri = df_all.set_index("hora")[[SERIES_TAXA["btc"], SERIES_TAXA["eth"]]].resample("QS").median()
tri = tri.iloc[1:-1] if len(tri) > 3 else tri
print("\nMediana trimestral da taxa (primeiro vs. último trimestre completo):")
display(pd.DataFrame({"primeiro": tri.iloc[0], "último": tri.iloc[-1], "razão": tri.iloc[0] / tri.iloc[-1]}).round(3))

## 2. Correções às leituras do 04

### 2.1 Fear & Greed × taxa: espúrio por tendência comum?

As correlações de ρ = +0,64 e +0,74 entre o F&G e a taxa **em nível** coincidem com uma queda estrutural da taxa. Se F&G e taxa têm ambos tendência, a correlação é artefato. Aqui compara-se (i) o nível, (ii) a taxa **relativa à mediana de 30 dias**, (iii) as **primeiras diferenças diárias** e (iv) a tendência de cada série contra o tempo.

In [ ]:
def log_pos(s):
    return np.log(s.where(s > 0))

dev = df_all[df_all["particao"] == "dev"].reset_index(drop=True)
hold = df_all[df_all["particao"] == "holdout"].reset_index(drop=True)
assert max(f["te_fim"] for f in folds) <= len(dev), "folds incompatíveis com o dev do dataset"

dd = dev.assign(dia=dev["hora"].dt.floor("D")).groupby("dia").agg(
    fng=("fng_lag1", "first"), btc_nivel=("btc_lfee_p50", "mean"), btc_rel=("btc_lfee_rel30d", "mean"),
    eth_nivel=("eth_lbase", "mean"), eth_rel=("eth_lbase_rel30d", "mean")).dropna()
dd["t"] = np.arange(len(dd))
linhas = []
for rede, nivel, rel in (("BTC", "btc_nivel", "btc_rel"), ("ETH", "eth_nivel", "eth_rel")):
    for nome, x, y in [("F&G vs taxa em NÍVEL", dd["fng"], dd[nivel]),
                       ("F&G vs taxa RELATIVA (vs. mediana 30d)", dd["fng"], dd[rel]),
                       ("Δ F&G vs Δ taxa (primeiras diferenças)", dd["fng"].diff(), dd[nivel].diff()),
                       ("tendência: F&G vs tempo", dd["fng"], dd["t"]),
                       ("tendência: taxa vs tempo", dd[nivel], dd["t"])]:
        m = x.notna() & y.notna()
        rho, p = stats.spearmanr(x[m], y[m])
        linhas.append({"rede": rede, "relação": nome, "n_dias": int(m.sum()), "Spearman": rho, "p_valor": p})
corr_fng = pd.DataFrame(linhas)
display(corr_fng.round(4))
for rede in ("BTC", "ETH"):
    a = corr_fng[(corr_fng["rede"] == rede)].set_index("relação")["Spearman"]
    print(f"{rede}: |ρ| nível = {abs(a['F&G vs taxa em NÍVEL']):.2f} | |ρ| relativa = {abs(a['F&G vs taxa RELATIVA (vs. mediana 30d)']):.2f} "
          f"| |ρ| diferenças = {abs(a['Δ F&G vs Δ taxa (primeiras diferenças)']):.2f}")

### 2.2 Deriva do evento relativo e relevância econômica

`ev_x` exige apenas `2 × mediana de 7 dias`. Com a taxa no piso, isso significa subir centavos. A célula mede (i) a tendência da prevalência de cada evento por trimestre e (ii) o aumento **absoluto** que cada definição exige no final do período de desenvolvimento.

In [ ]:
prev = df_all.set_index("hora")[[c for c in df_all.columns if c.startswith("y_") and "_ev_" in c]].resample("QS").mean()
prev = prev.iloc[:-1] if len(prev) > 2 else prev           # descarta trimestre parcial final
linhas = []
for c in prev.columns:
    rho, p = stats.spearmanr(np.arange(len(prev)), prev[c].to_numpy())
    linhas.append({"evento": c, "prev_min": prev[c].min(), "prev_max": prev[c].max(), "amplitude": prev[c].max() - prev[c].min(),
                   "tendência (ρ vs tempo)": rho})
deriva = pd.DataFrame(linhas)
print("Estabilidade da prevalência por trimestre (quanto maior |ρ|/amplitude, mais o evento deriva):")
display(deriva.round(3))

ult = dev.iloc[-90 * 24:]
linhas = []
for a, conv in (("btc", lambda inc, p: inc * 141 / 1e8 * p), ("eth", lambda inc, p: inc * 21000 / 1e9 * p)):
    ref_med = ult[f"{a}_taxa_ref_med7d"]
    preco = ult[f"{a}_fechamento"]
    inc_x = 1.0 * ref_med                                    # ev_x: 2x - 1x = 1x a mediana
    inc_q = (ult[f"{a}_taxa_ref_q"] - ref_med).clip(lower=0)  # ev_q: distância mediana -> P90 móvel
    linhas.append({"rede": a.upper(), "evento": "ev_x (≥2x mediana 7d)", "aumento_mediano_taxa": inc_x.median(),
                   "aumento_mediano_USD": conv(inc_x, preco).median()})
    linhas.append({"rede": a.upper(), "evento": "ev_q (> P90 móvel 30d)", "aumento_mediano_taxa": inc_q.median(),
                   "aumento_mediano_USD": conv(inc_q, preco).median()})
print("\nAumento absoluto exigido para disparar o evento (últimos 90 dias do DEV; taxa em sat/vB ou gwei):")
display(pd.DataFrame(linhas).round(5))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, a in zip(axes, ("btc", "eth")):
    for tipo, cor in (("q", "#1b9e77"), ("x", "#d95f02")):
        c = f"y_{a}_ev_{tipo}_6h"
        ax.plot(prev.index, prev[c], "o-", color=cor, label=f"ev_{tipo} (6h)")
    ax.set_title(f"{a.upper()}: prevalência do evento por trimestre"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Variáveis novas (causais) e teste de perturbação

- **Estado da própria taxa:** distância ao limiar móvel (`dist_ref_q`), indicador de que a taxa já está em evento (`em_evento`), p90 relativo, prioridade relativa (ETH) e média móvel do uso de gás.
- **Estrutura/atividade:** ocupação do bloco BTC (inclusive o indicador de mempool drenada `ocup < 0,95`), contagem de transações e vsize **relativos** a 7 dias.
- **Choque de volatilidade:** `log RV − log RV semanal`.
- **Fluxo de ordens (substituto do sentimento textual):** desequilíbrio de compra agressiva da Binance (`taker_ratio − 0,5`), média de 24 h e intensidade de 6 h.
- **Alvo de início de evento (`novo`):** o evento ocorre em `t+1…t+h` **e a taxa ainda não estava em evento em `t`**. Remove a persistência trivial e é o que importa para decidir entre enviar agora ou esperar. É avaliado só nas horas em que a taxa está em estado normal.

In [ ]:
def construir_extras(df):
    d = df.copy()
    for a in ("btc", "eth"):
        x = d[SERIES_TAXA[a]]
        ref = d[f"{a}_taxa_ref_q"]
        d[f"{a}_dist_ref_q"] = log_pos(x) - log_pos(ref)
        d[f"{a}_em_evento"] = (x > ref).astype(float).where(x.notna() & ref.notna())
        d[f"{a}_lrv_choque"] = d[f"{a}_lrv"] - d[f"{a}_lrv_w"]
        col = f"{a}_taker_ratio"
        if col in d.columns:
            imb = d[col] - 0.5
            d[f"{a}_taker_imb"] = imb
            d[f"{a}_taker_imb_24h"] = imb.rolling(24, min_periods=18).mean()
            d[f"{a}_taker_int_6h"] = imb.abs().rolling(6, min_periods=4).mean()
    p90 = d["btc_bq_taxa_p90_sat_vb"]
    d["btc_lfee_p90_rel"] = log_pos(p90) - log_pos(p90.rolling(720, min_periods=360).median())
    d["btc_ocup"] = d["btc_bq_ocupacao_media"]
    d["btc_ocup_lt95"] = (d["btc_ocup"] < 0.95).astype(float).where(d["btc_ocup"].notna())
    d["btc_ocup_24h"] = d["btc_ocup"].rolling(24, min_periods=18).mean()
    for nome, col in (("btc_ltx_rel", "btc_bq_tx"), ("btc_lvsize_rel", "btc_bq_vsize_total"), ("eth_ltx_rel", "eth_bq_tx")):
        s = d[col]
        d[nome] = log_pos(s) - log_pos(s.rolling(168, min_periods=126).mean())
    prio = d["eth_bq_preco_gas_p50_gwei"]
    d["eth_lprio_rel"] = log_pos(prio) - log_pos(prio.rolling(720, min_periods=360).median())
    d["eth_gas_ratio_c_6h"] = d["eth_gas_ratio_c"].rolling(6, min_periods=4).mean()
    # alvos de início de evento: evento futuro E taxa normal em t
    for a in ("btc", "eth"):
        for h in HORIZONTES_TAXA:
            ev = d[f"y_{a}_ev_q_{h}h"]
            d[f"y_{a}_novo_{h}h"] = ev.where(d[f"{a}_em_evento"] == 0)
    return d

extras_antes = set(df_all.columns)
df_all = construir_extras(df_all)
cols_novas = [c for c in df_all.columns if c not in extras_antes]
cols_novas_feat = [c for c in cols_novas if not c.startswith("y_")]

# ---- teste de perturbação: lixo em todos os dados depois de t0 -> features novas antes de t0 não podem mudar ----
def teste_perturbacao_extras(df, t0):
    d2 = df.copy()
    num = [c for c in d2.select_dtypes("number").columns]
    d2[num] = d2[num].astype(float)
    pos = [d2.columns.get_loc(c) for c in num]
    d2.iloc[t0 + 1:, pos] = d2.iloc[t0 + 1:, pos].to_numpy() * rng.uniform(0.5, 2.0, size=(len(d2) - t0 - 1, len(pos)))
    base_cols = [c for c in df.columns if c not in cols_novas]
    e1 = construir_extras(df[base_cols]); e2 = construir_extras(d2[base_cols])
    ruins = [c for c in cols_novas_feat if not np.allclose(e1[c].iloc[:t0 + 1].to_numpy(float), e2[c].iloc[:t0 + 1].to_numpy(float), equal_nan=True)]
    return ruins

t0 = len(df_all) // 2
vazadas = teste_perturbacao_extras(df_all, t0)
print(f"Features novas: {len(cols_novas_feat)} | alvos novos: {len([c for c in cols_novas if c.startswith('y_')])}")
print(f"Teste de perturbação em t0 = {df_all['hora'].iloc[t0]}: com vazamento = {vazadas}")
assert not vazadas, f"VAZAMENTO nas features: {vazadas}"

dev = df_all[df_all["particao"] == "dev"].reset_index(drop=True)
hold = df_all[df_all["particao"] == "holdout"].reset_index(drop=True)

cols_ev = [f"y_{a}_{t}_{h}h" for a in ("btc", "eth") for t in ("ev_q", "novo") for h in HORIZONTES_TAXA]
print("\nPrevalência dos alvos de taxa no DEV (novo = só horas em estado normal):")
display(pd.DataFrame({"prevalência": dev[cols_ev].mean(), "n_linhas_validas": dev[cols_ev].notna().sum()}).round(4))

### 3.1 Ocupação do bloco BTC como indicador de fila

A ocupação mediana é ~0,998, mas a cauda inferior (mempool drenada) pode informar. Compara-se a taxa de **início de evento** nas próximas horas quando a ocupação está abaixo de 0,95 (fila vazia) e quando não está.

In [ ]:
linhas = []
for h in HORIZONTES_TAXA:
    y = dev[f"y_btc_novo_{h}h"]
    for rot, mask in (("ocupação < 0,95 (mempool drenada)", dev["btc_ocup_lt95"] == 1), ("ocupação ≥ 0,95", dev["btc_ocup_lt95"] == 0)):
        m = mask & y.notna()
        linhas.append({"h": h, "grupo": rot, "n": int(m.sum()), "P(início de evento)": y[m].mean()})
tb = pd.DataFrame(linhas)
display(tb.round(4))
for h in HORIZONTES_TAXA:
    t = tb[(tb["h"] == h) & (tb["n"] > 0)].set_index("grupo")["P(início de evento)"]
    if len(t) == 2:
        print(f"h={h}h: razão P(drenada)/P(cheia) = {t.iloc[0] / t.iloc[1]:.2f}")
    else:
        print(f"h={h}h: um dos grupos está vazio; razão não calculável.")

## 4. Utilidades estatísticas

Regressão logística com L2 em NumPy (sem dependência externa), métricas de classificação probabilística (PR-AUC, AUC, Brier, *skill* de Brier, log-loss, lift no top-10%), Diebold–Mariano com Newey–West, correção de Holm e *bootstrap* em blocos.

In [ ]:
def ols_hac(y, X, L):
    X = np.column_stack([np.ones(len(y)), X]) if X.size else np.ones((len(y), 1))
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XtX_inv = np.linalg.inv(X.T @ X)
    Xe = X * e[:, None]
    S = Xe.T @ Xe
    for l in range(1, L + 1):
        w = 1 - l / (L + 1)
        G = Xe[l:].T @ Xe[:-l]
        S += w * (G + G.T)
    return beta, XtX_inv @ S @ XtX_inv

def dm_hac(d, L):
    beta, V = ols_hac(d, np.empty((len(d), 0)), L)
    t = beta[0] / np.sqrt(V[0, 0])
    return float(t), float(2 * stats.norm.sf(abs(t)))

def holm(p):
    p = np.asarray(p, float)
    ordem = np.argsort(p); ajust = np.empty(len(p)); corrente = 0.0
    for rank, i in enumerate(ordem):
        corrente = max(corrente, (len(p) - rank) * p[i]); ajust[i] = min(1.0, corrente)
    return ajust

def qlike(r, f):
    q = r / f
    return q - np.log(q) - 1

def ols_fit(X, y):
    A = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    return beta, float((y - A @ beta).var(ddof=A.shape[1]))

def ols_pred(beta, X):
    return np.column_stack([np.ones(len(X)), X]) @ beta

def logit_fit(X, y, l2=L2_LOGIT, iters=60, tol=1e-6):
    mu, sd = X.mean(0), X.std(0)
    sd = np.where(sd == 0, 1.0, sd)
    A = np.column_stack([np.ones(len(X)), (X - mu) / sd])
    beta = np.zeros(A.shape[1])
    P = np.eye(A.shape[1]) * l2; P[0, 0] = 0
    for _ in range(iters):
        p = 1 / (1 + np.exp(-np.clip(A @ beta, -30, 30)))
        W = np.clip(p * (1 - p), 1e-6, None)
        g = A.T @ (y - p) - P @ beta
        H = (A * W[:, None]).T @ A + P
        step = np.linalg.solve(H, g)
        beta += step
        if np.max(np.abs(step)) < tol:
            break
    return beta, mu, sd

def logit_pred(m, X):
    beta, mu, sd = m
    A = np.column_stack([np.ones(len(X)), (X - mu) / sd])
    return 1 / (1 + np.exp(-np.clip(A @ beta, -30, 30)))

def ap_score(y, p):
    y = np.asarray(y, float)
    if y.sum() == 0:
        return np.nan
    ys = y[np.argsort(-np.asarray(p, float), kind="mergesort")]
    prec = np.cumsum(ys) / np.arange(1, len(ys) + 1)
    return float((prec * ys).sum() / y.sum())

def auc_score(y, p):
    y = np.asarray(y, float); n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = stats.rankdata(p)
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def logloss(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def lift_top(y, p, frac=0.10):
    k = max(1, int(len(y) * frac))
    return float(y[np.argsort(-p)[:k]].mean() / y.mean()) if y.mean() > 0 else np.nan

def metricas(y, p, p0):
    brier = float(np.mean((p - y) ** 2)); brier0 = float(np.mean((p0 - y) ** 2))
    return {"prev": float(y.mean()), "AP": ap_score(y, p), "AUC": auc_score(y, p), "Brier": brier,
            "BSS": 1 - brier / brier0, "LogLoss": logloss(y, p), "Lift_top10": lift_top(y, p)}

def blocos_idx(n, B, bloco=BLOCO_H):
    nb = int(np.ceil(n / bloco))
    starts = rng.integers(0, n - bloco + 1, size=(B, nb)).astype(np.int32)
    return (starts[:, :, None] + np.arange(bloco, dtype=np.int32)).reshape(B, -1)[:, :n]

print("Utilidades prontas.")

## 5. Refinamento do Modelo A (volatilidade)

No 04, on-chain em **nível** e Fear & Greed pioraram o HAR. Aqui testa-se se a piora vinha da não estacionariedade, trocando as variáveis de nível por **relativas**, e se o **fluxo de ordens** agrega. Referência: M3 (HAR + calendário + mercado). Comparações por Diebold–Mariano (HAC) com Holm.

Escolha do modelo final do Modelo A por **regra fixa**: o mais simples cujo QLIKE médio em `dev` fique a até 1% do melhor.

In [ ]:
def conj_vol(a):
    har = [f"{a}_lrv", f"{a}_lrv_d", f"{a}_lrv_w"]
    cal = ["cal_sin", "cal_cos", "cal_fim_semana"]
    mkt = [f"{a}_ret_1h", f"{a}_ret_neg_24h", f"{a}_lvol_rel"]
    flux = [c for c in (f"{a}_taker_imb", f"{a}_taker_imb_24h", f"{a}_taker_int_6h") if c in df_all.columns]
    if a == "btc":
        rel = ["btc_lfee_rel30d", "btc_lfee_chg6", "btc_lfee_p90_rel", "btc_ocup_lt95", "btc_hora_sem_bloco"]
    else:
        rel = ["eth_lbase_rel30d", "eth_lbase_chg6", "eth_gas_ratio_c", "eth_gas_ratio_c_6h", "eth_lprio_rel"]
    c = {"M1": har, "M2": har + cal, "M3": har + cal + mkt, "M3+fluxo": har + cal + mkt + flux,
         "M3+on-chain rel.": har + cal + mkt + rel, "M3+fluxo+on-chain rel.": har + cal + mkt + flux + rel}
    return {k: v for k, v in c.items() if k != "M3+fluxo" or flux}

ORDEM_COMPLEXIDADE_A = ["M1", "M2", "M3", "M3+fluxo", "M3+on-chain rel.", "M3+fluxo+on-chain rel."]

def rodar_vol(a, h):
    ycol = f"y_{a}_lrv_{h}h"
    cj = conj_vol(a)
    todas = sorted(set(sum(cj.values(), [])))
    regs = []
    for k, f in enumerate(folds):
        tr = dev.iloc[f["tr_ini"]:f["tr_fim"]].dropna(subset=todas + [ycol])
        te = dev.iloc[f["te_ini"]:f["te_fim"]].dropna(subset=todas + [ycol])
        reg = {"hora": te["hora"].to_numpy(), "fold": k + 1, "y": te[ycol].to_numpy()}
        for nome, cols in cj.items():
            beta, s2 = ols_fit(tr[cols].to_numpy(float), tr[ycol].to_numpy(float))
            reg[f"yhat_{nome}"] = ols_pred(beta, te[cols].to_numpy(float)); reg[f"s2_{nome}"] = s2
        regs.append(pd.DataFrame(reg))
    P = pd.concat(regs, ignore_index=True)
    r = np.exp(2 * P["y"].to_numpy())
    for nome in cj:
        P[f"ql_{nome}"] = qlike(r, np.exp(2 * P[f"yhat_{nome}"].to_numpy() + 2 * P[f"s2_{nome}"].to_numpy()))
    return P, list(cj)

linhas_ql, linhas_inc, escolha_A = [], [], {}
for a in ("btc", "eth"):
    for h in HORIZONTES_VOL:
        P, nomes = rodar_vol(a, h)
        ql = {n: P[f"ql_{n}"].mean() for n in nomes}
        linhas_ql.append({"rede": a.upper(), "h": h, **{f"QLIKE_{n}": v for n, v in ql.items()}})
        melhor = min(ql.values())
        escolha_A[(a, h)] = next(n for n in ORDEM_COMPLEXIDADE_A if n in ql and ql[n] <= melhor * 1.01)
        for nome in nomes:
            if nome in ("M1", "M2", "M3"):
                continue
            d = (P["ql_M3"] - P[f"ql_{nome}"]).to_numpy()
            t, p = dm_hac(d, L=max(24, h) + h)
            n_f = int(P.groupby("fold").apply(lambda g: g["ql_M3"].mean() > g[f"ql_{nome}"].mean()).sum())
            linhas_inc.append({"rede": a.upper(), "h": h, "modelo vs M3": nome, "ganho_QLIKE_%": 100 * d.mean() / P["ql_M3"].mean(),
                               "DM_t": t, "p_valor": p, "folds_melhores": f"{n_f}/{P['fold'].nunique()}"})
tab_vol = pd.DataFrame(linhas_ql)
inc_vol = pd.DataFrame(linhas_inc)
inc_vol["p_Holm"] = holm(inc_vol["p_valor"].to_numpy())
print("QLIKE médio (menor = melhor):")
display(tab_vol.round(4))
print("\nGanho vs M3 (QLIKE > 0 = melhora), Diebold-Mariano (HAC) com Holm:")
display(inc_vol.round(4))
print("\nModelo A escolhido pela regra de parcimônia (por rede e horizonte):")
display(pd.DataFrame([{"rede": a.upper(), "h": h, "modelo": m} for (a, h), m in escolha_A.items()]).pivot(index="rede", columns="h", values="modelo"))

## 6. Modelo B: probabilidade de evento de taxa

**Escada de informação** (cada degrau acrescenta um bloco):

| Degrau | Variáveis |
|---|---|
| B0 | taxa-base do treino (sem variáveis) |
| B1 | **estado da própria taxa** (distância ao limiar, estado atual, variação, p90/prioridade relativos, uso de gás) + calendário |
| B2 | + **mercado** (RV, choque de RV, retornos, volume relativo) |
| B3 | + **fluxo de ordens** e **estrutura on-chain** (ocupação, transações, vsize relativos) |
| B4 | + **Fear & Greed** |

Dois alvos por rede e horizonte: `ev_q` (todas as horas) e `novo` (início de evento, só horas em estado normal). O B2 sobre o B1 responde à pergunta central: **o mercado agrega à previsão da taxa além da própria taxa?** Modelos: logística L2 (todos os degraus) e *gradient boosting* (B1 e B4, para checar não linearidade).

Avaliação: PR-AUC (referência = prevalência), AUC, Brier, *skill* de Brier (vs. taxa-base), log-loss, lift no top-10%. Ganho incremental: DM sobre o Brier (HAC) e ΔPR-AUC com *bootstrap* em blocos de 48 h; Holm sobre todos os testes.

In [ ]:
def grupos_B(a):
    cal = ["cal_sin", "cal_cos", "cal_fim_semana"]
    if a == "btc":
        estado = ["btc_lfee_rel30d", "btc_lfee_chg6", "btc_dist_ref_q", "btc_em_evento", "btc_lfee_p90_rel"]
        estrutura = ["btc_ocup", "btc_ocup_lt95", "btc_ocup_24h", "btc_hora_sem_bloco", "btc_ltx_rel", "btc_lvsize_rel"]
    else:
        estado = ["eth_lbase_rel30d", "eth_lbase_chg6", "eth_dist_ref_q", "eth_em_evento", "eth_gas_ratio_c", "eth_gas_ratio_c_6h", "eth_lprio_rel"]
        estrutura = ["eth_ltx_rel"]
    mercado = [f"{a}_lrv", f"{a}_lrv_d", f"{a}_lrv_w", f"{a}_lrv_choque", f"{a}_ret_1h", f"{a}_ret_neg_24h", f"{a}_lvol_rel"]
    fluxo = [c for c in (f"{a}_taker_imb", f"{a}_taker_imb_24h", f"{a}_taker_int_6h") if c in df_all.columns]
    sent = ["fng_lag1", "fng_var7"]
    g = {"B1": estado + cal, "B2": estado + cal + mercado, "B3": estado + cal + mercado + fluxo + estrutura,
         "B4": estado + cal + mercado + fluxo + estrutura + sent}
    return g

def alvo_col(a, tipo, h):
    return f"y_{a}_ev_q_{h}h" if tipo == "ev_q" else f"y_{a}_novo_{h}h"

def novo_gb():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0,
                                          early_stopping=False, random_state=SEMENTE)

def rodar_B(a, tipo, h, familia):
    ycol = alvo_col(a, tipo, h)
    g = grupos_B(a)
    todas = sorted(set(g["B4"]))
    if tipo == "novo":                                       # nas linhas 'novo' o estado é sempre 0: remove colunas constantes
        g = {k: [c for c in v if c != f"{a}_em_evento"] for k, v in g.items()}
        todas = sorted(set(g["B4"]))
    degraus = list(g) if familia == "logit" else ["B1", "B4"]
    regs = []
    for k, f in enumerate(folds):
        tr = dev.iloc[f["tr_ini"]:f["tr_fim"]].dropna(subset=todas + [ycol])
        te = dev.iloc[f["te_ini"]:f["te_fim"]].dropna(subset=todas + [ycol])
        if tr[ycol].sum() < 20 or len(te) == 0:
            continue
        ytr = tr[ycol].to_numpy(float)
        reg = {"pos": te.index.to_numpy(), "hora": te["hora"].to_numpy(), "fold": k + 1, "y": te[ycol].to_numpy(float),
               "p_B0": np.full(len(te), ytr.mean())}
        for nome in degraus:
            cols = g[nome]
            if familia == "logit":
                m = logit_fit(tr[cols].to_numpy(float), ytr)
                reg[f"p_{nome}"] = logit_pred(m, te[cols].to_numpy(float))
                p_in = logit_pred(m, tr[cols].to_numpy(float))
                for r in TAXAS_ALERTA:                          # limiar de alerta fixado só com o treino
                    reg[f"tau_{nome}_{r}"] = np.full(len(te), np.quantile(p_in, 1 - r))
            else:
                gb = novo_gb().fit(tr[cols].to_numpy(float), ytr)
                reg[f"p_{nome}"] = gb.predict_proba(te[cols].to_numpy(float))[:, 1]
        regs.append(pd.DataFrame(reg))
    return (pd.concat(regs, ignore_index=True) if regs else None), degraus

def tabela_incremental(P, degraus, h, rotulo):
    pares = [(a_, b_) for a_, b_ in zip(["B0"] + degraus[:-1], degraus)] if len(degraus) > 2 else [("B0", degraus[0]), (degraus[0], degraus[-1])]
    if len(degraus) > 2:
        pares.append((degraus[0], degraus[-1]))
    y = P["y"].to_numpy(float)
    idx = blocos_idx(len(y), B_BOOT)
    out = []
    for ant, nov in pares:
        pa, pn = P[f"p_{ant}"].to_numpy(float), P[f"p_{nov}"].to_numpy(float)
        d = (pa - y) ** 2 - (pn - y) ** 2
        t, p = dm_hac(d, L=max(24, h) + h)
        yb = y[idx]; pab = pa[idx]; pnb = pn[idx]
        dap = np.array([ap_score(yb[b], pnb[b]) - ap_score(yb[b], pab[b]) for b in range(B_BOOT)])
        dap_obs = ap_score(y, pn) - ap_score(y, pa)
        out.append({**rotulo, "degrau": f"{ant} -> {nov}", "dBrier_%": 100 * d.mean() / np.mean((pa - y) ** 2),
                    "DM_t": t, "p_valor": p, "dAP": dap_obs, "dAP_IC95_inf": np.nanpercentile(dap, 2.5),
                    "dAP_IC95_sup": np.nanpercentile(dap, 97.5)})
    return out

resultados_B, linhas_met, linhas_inc_B, preds_B = {}, [], [], {}
for a in ("btc", "eth"):
    for tipo in ("ev_q", "novo"):
        for h in HORIZONTES_TAXA:
            for fam in (["logit", "gb"] if TEM_SKLEARN else ["logit"]):
                P, deg = rodar_B(a, tipo, h, fam)
                if P is None:
                    print(f"[{a} {tipo} {h}h {fam}] sem dados suficientes"); continue
                preds_B[(a, tipo, h, fam)] = (P, deg)
                y = P["y"].to_numpy(float)
                for nome in ["B0"] + deg:
                    linhas_met.append({"rede": a.upper(), "alvo": tipo, "h": h, "modelo": fam, "degrau": nome, "n": len(y),
                                       **metricas(y, P[f"p_{nome}"].to_numpy(float), P["p_B0"].to_numpy(float))})
                linhas_inc_B += tabela_incremental(P, deg, h, {"rede": a.upper(), "alvo": tipo, "h": h, "modelo": fam})

met_B = pd.DataFrame(linhas_met)
inc_B = pd.DataFrame(linhas_inc_B)
inc_B["p_Holm"] = holm(inc_B["p_valor"].to_numpy())
print("Métricas fora da amostra (folds walk-forward de DEV, previsões empilhadas):")
display(met_B.round(4))

In [ ]:
print("Ganho incremental de cada degrau (dBrier_% > 0 = melhora). p_Holm corrige todos os testes juntos:")
display(inc_B.round(4))

# veredito calculado
def veredito(r):
    if r["p_Holm"] < 0.05 and r["dBrier_%"] > 0:
        return "melhora robusta"
    if r["p_valor"] < 0.05 and r["dBrier_%"] > 0:
        return "melhora (não resiste ao Holm)"
    if r["p_Holm"] < 0.05 and r["dBrier_%"] < 0:
        return "PIORA significativa"
    return "sem efeito detectável"
inc_B["veredito"] = inc_B.apply(veredito, axis=1)
print("\nContagem de vereditos por degrau (logística):")
display(inc_B[inc_B["modelo"] == "logit"].groupby(["degrau", "veredito"]).size().unstack(fill_value=0))

# skill de Brier por degrau
lg = met_B[met_B["modelo"] == "logit"]
combos = lg[["rede", "alvo", "h"]].drop_duplicates().to_records(index=False)
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for ax, (rede, alvo, h) in zip(axes.ravel(), combos):
    s = lg[(lg["rede"] == rede) & (lg["alvo"] == alvo) & (lg["h"] == h) & (lg["degrau"] != "B0")]
    ax.bar(s["degrau"], s["BSS"], color="#1b9e77"); ax.axhline(0, color="k", lw=0.6)
    ax.set_title(f"{rede} {alvo} {h}h"); ax.set_ylabel("skill de Brier vs. taxa-base" if ax in axes[:, 0] else "")
plt.tight_layout(); plt.show()

### 6.1 Calibração e escolha dos modelos (regra fixa)

Para cada (rede, alvo, horizonte) escolhe-se o **degrau mais simples cujo Brier fique a até 1% do melhor** (logística, folds de `dev`). Isso congela as especificações antes de qualquer olhada no *holdout*. O gráfico mostra a curva de calibração do degrau escolhido.

In [ ]:
ORDEM_B = ["B1", "B2", "B3", "B4"]
escolha_B = {}
for (a, tipo, h, fam), (P, deg) in preds_B.items():
    if fam != "logit":
        continue
    y = P["y"].to_numpy(float)
    br = {n: float(np.mean((P[f"p_{n}"].to_numpy(float) - y) ** 2)) for n in deg}
    melhor = min(br.values())
    escolha_B[(a, tipo, h)] = next(n for n in ORDEM_B if n in br and br[n] <= melhor * 1.01)
print("Degrau escolhido (Modelo B):")
display(pd.DataFrame([{"rede": a.upper(), "alvo": t, "h": h, "degrau": d} for (a, t, h), d in escolha_B.items()])
        .pivot(index=["rede", "alvo"], columns="h", values="degrau"))

fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharex=True, sharey=True)
for ax, ((a, tipo, h), nome) in zip(axes.ravel(), escolha_B.items()):
    P, _ = preds_B[(a, tipo, h, "logit")]
    p = P[f"p_{nome}"].to_numpy(float); y = P["y"].to_numpy(float)
    bins = np.unique(np.quantile(p, np.linspace(0, 1, 11)))
    cat = np.clip(np.digitize(p, bins[1:-1]), 0, len(bins) - 2)
    xs = [p[cat == c].mean() for c in np.unique(cat)]; ys = [y[cat == c].mean() for c in np.unique(cat)]
    ax.plot([0, max(xs + ys)], [0, max(xs + ys)], "k--", lw=0.8); ax.plot(xs, ys, "o-", color="#d95f02")
    ax.set_title(f"{a.upper()} {tipo} {h}h ({nome})"); ax.set_xlabel("prob. prevista"); ax.set_ylabel("freq. observada")
plt.tight_layout(); plt.show()

## 7. Utilidade operacional: quando vale esperar?

Decisão a cada hora, **em qualquer estado**: uma transação precisa ser executada em até **6 horas**. Em estado normal, o custo já é baixo e "enviar agora" é quase sempre a melhor ação; o valor de prever está em saber **quando esperar compensa** (taxa alta que vai cair). Por isso o alvo é contínuo:

$$\Delta_t = \ln c_{t+6} - \ln c_{t+1}$$

(log da razão entre o custo da transferência padrão no prazo e o custo imediato). Regressão linear com os mesmos degraus B1 a B4 (walk-forward, só `dev`).

| Política | Regra |
|---|---|
| agora | executa na hora seguinte (`t+1`) |
| prazo | espera e executa no limite (`t+6`) |
| modelo | **espera até o prazo só nas horas em que $\hat\Delta_t$ prevê as quedas mais fortes** (5%, 10% ou 20% das horas, com limiar fixado só no treino de cada *fold*); nas demais, executa agora |
| oráculo | executa na hora mais barata das próximas 6 (limite inatingível, só referência) |

Custo relativo = custo da transferência padrão ÷ mediana causal de 168 h. Intervalos por *bootstrap* em blocos. A economia em dólares é pequena (o custo é de centavos); o relevante é a razão e o efeito em escala.

In [ ]:
def alvo_delta(a, h=H_PRAZO):
    c = dev[f"{a}_custo_usd"]
    return log_pos(c.shift(-h)) - log_pos(c.shift(-1))

def rodar_delta(a):
    g = grupos_B(a)
    todas = sorted(set(g["B4"]))
    d_all = dev.copy()
    d_all["delta"] = alvo_delta(a)
    regs = []
    for k, f in enumerate(folds):
        tr = d_all.iloc[f["tr_ini"]:f["tr_fim"]].dropna(subset=todas + ["delta"])
        te = d_all.iloc[f["te_ini"]:f["te_fim"]].dropna(subset=todas + ["delta"])
        if len(tr) < 500 or len(te) == 0:
            continue
        ytr = tr["delta"].to_numpy(float)
        reg = {"pos": te.index.to_numpy(), "hora": te["hora"].to_numpy(), "fold": k + 1, "y": te["delta"].to_numpy(float),
               "d_B0": np.full(len(te), ytr.mean())}
        for nome, cols in g.items():
            beta, _ = ols_fit(tr[cols].to_numpy(float), ytr)
            reg[f"d_{nome}"] = ols_pred(beta, te[cols].to_numpy(float))
            p_in = ols_pred(beta, tr[cols].to_numpy(float))
            for r in TAXAS_ALERTA:
                reg[f"tau_{nome}_{r}"] = np.full(len(te), np.quantile(p_in, r))      # cauda de quedas previstas
        regs.append(pd.DataFrame(reg))
    return pd.concat(regs, ignore_index=True), list(g)

preds_delta, linhas_d, linhas_dinc = {}, [], []
for a in ("btc", "eth"):
    P, deg = rodar_delta(a)
    preds_delta[a] = (P, deg)
    y = P["y"].to_numpy(float)
    mse0 = float(np.mean((y - P["d_B0"].to_numpy(float)) ** 2))
    for nome in ["B0"] + deg:
        e = y - P[f"d_{nome}"].to_numpy(float)
        linhas_d.append({"rede": a.upper(), "degrau": nome, "n": len(y), "MSE": float(np.mean(e ** 2)),
                         "R2_oos_vs_média": 1 - float(np.mean(e ** 2)) / mse0,
                         "corr(prev, obs)": float(np.corrcoef(P[f"d_{nome}"], y)[0, 1]) if nome != "B0" else np.nan})
    for ant, nov in [("B0", "B1"), ("B1", "B2"), ("B2", "B3"), ("B3", "B4"), ("B1", "B4")]:
        d = (y - P[f"d_{ant}"].to_numpy(float)) ** 2 - (y - P[f"d_{nov}"].to_numpy(float)) ** 2
        t, p = dm_hac(d, L=max(24, H_PRAZO) + H_PRAZO)
        linhas_dinc.append({"rede": a.upper(), "degrau": f"{ant} -> {nov}", "dMSE_%": 100 * d.mean() / np.mean((y - P[f"d_{ant}"].to_numpy(float)) ** 2),
                            "DM_t": t, "p_valor": p})
tab_delta = pd.DataFrame(linhas_d)
inc_delta = pd.DataFrame(linhas_dinc)
inc_delta["p_Holm"] = holm(inc_delta["p_valor"].to_numpy())
print("Previsão de Δ = ln custo(t+6) − ln custo(t+1) (fora da amostra, folds de DEV):")
display(tab_delta.round(4))
print("\nGanho incremental (dMSE_% > 0 = melhora):")
display(inc_delta.round(4))

def simular_espera(a, rung, rate, P, h=H_PRAZO):
    cst = dev[f"{a}_custo_usd"].to_numpy(float)
    cref = dev[f"{a}_custo_usd"].rolling(168, min_periods=84).median().to_numpy(float)
    n = len(dev)
    pos = P["pos"].to_numpy(int)
    F = np.full((len(pos), h), np.nan)
    for k in range(1, h + 1):
        ok = pos + k < n
        F[ok, k - 1] = cst[pos[ok] + k]
    ref = cref[pos]
    valido = np.isfinite(F).all(axis=1) & np.isfinite(ref) & (ref > 0)
    F, ref = F[valido], ref[valido]
    espera = (P[f"d_{rung}"].to_numpy(float) <= P[f"tau_{rung}_{rate}"].to_numpy(float))[valido]
    agora, prazo, orac = F[:, 0], F[:, h - 1], F.min(axis=1)
    modelo = np.where(espera, prazo, agora)
    rel_ag, rel_pr, rel_mo, rel_or = agora / ref, prazo / ref, modelo / ref, orac / ref
    idx = blocos_idx(len(F), B_BOOT)
    sv_ag = np.array([rel_ag[i].mean() - rel_mo[i].mean() for i in idx])        # >0: modelo mais barato que 'agora'
    sv_pr = np.array([rel_pr[i].mean() - rel_mo[i].mean() for i in idx])        # >0: modelo mais barato que 'prazo'
    return {"n": int(len(F)), "espera_%": 100 * espera.mean(), "custo_rel_agora": rel_ag.mean(), "custo_rel_prazo": rel_pr.mean(),
            "custo_rel_modelo": rel_mo.mean(), "custo_rel_oraculo": rel_or.mean(),
            "econ_vs_agora": rel_ag.mean() - rel_mo.mean(), "IC95_agora": (np.percentile(sv_ag, 2.5), np.percentile(sv_ag, 97.5)),
            "econ_vs_prazo": rel_pr.mean() - rel_mo.mean(), "IC95_prazo": (np.percentile(sv_pr, 2.5), np.percentile(sv_pr, 97.5)),
            "USD_por_tx_agora": agora.mean(), "USD_por_tx_modelo": modelo.mean()}

linhas = []
for a, (P, deg) in preds_delta.items():
    for rung in deg:
        for rate in TAXAS_ALERTA:
            linhas.append({"rede": a.upper(), "degrau": rung, "espera_alvo": rate, **simular_espera(a, rung, rate, P)})
sim = pd.DataFrame(linhas)
sim["IC95_agora"] = sim["IC95_agora"].apply(lambda t: f"[{t[0]:.3f}, {t[1]:.3f}]")
sim["IC95_prazo"] = sim["IC95_prazo"].apply(lambda t: f"[{t[0]:.3f}, {t[1]:.3f}]")
print("\nCusto relativo (1,00 = mediana recente). econ_* > 0 = a política do modelo é mais barata:")
display(sim.round(4))

# degrau escolhido para a política (regra fixa: mais simples a até 1% do menor MSE)
escolha_P = {}
for a in ("btc", "eth"):
    t = tab_delta[(tab_delta["rede"] == a.upper()) & (tab_delta["degrau"] != "B0")].set_index("degrau")["MSE"]
    escolha_P[a] = next(n for n in ORDEM_B if n in t.index and t[n] <= t.min() * 1.01)
TAXA_PRIMARIA = 0.10
print("\nDegrau escolhido para a política:", {a.upper(): d for a, d in escolha_P.items()}, "| taxa de espera primária:", TAXA_PRIMARIA)

## 8. *Holdout*: abertura única, com as especificações congeladas

Só roda se `ABRIR_HOLDOUT = True` (primeira célula). As escolhas vêm das regras fixas das seções 5 e 6.1; o treino usa **todo o `dev`**; o *holdout* nunca participou de nenhuma decisão. Cria um arquivo de trava: uma segunda abertura é recusada, a menos que `FORCAR_REABERTURA = True` (o que invalida o rigor do teste e deve ser declarado).

In [ ]:
LOCK = PASTA_SAIDA / "HOLDOUT_ABERTO.json"

def avaliar_holdout():
    # ---- Modelo A ----
    resA = []
    for (a, h), nome in escolha_A.items():
        cj = conj_vol(a)
        ycol = f"y_{a}_lrv_{h}h"
        usar = sorted(set(cj[nome] + cj["M1"] + cj["M2"] + [ycol]))
        tr = dev.dropna(subset=usar); te = hold.dropna(subset=usar)
        r = np.exp(2 * te[ycol].to_numpy(float))
        ql = {}
        for rot, cols in (("escolhido", cj[nome]), ("M1_HAR", cj["M1"]), ("M2", cj["M2"])):
            beta, s2 = ols_fit(tr[cols].to_numpy(float), tr[ycol].to_numpy(float))
            ql[rot] = qlike(r, np.exp(2 * ols_pred(beta, te[cols].to_numpy(float)) + 2 * s2))
        d = ql["M1_HAR"] - ql["escolhido"]
        t, p = dm_hac(d, L=max(24, h) + h)
        resA.append({"rede": a.upper(), "h": h, "modelo_escolhido": nome, "n": len(te), "QLIKE_escolhido": ql["escolhido"].mean(),
                     "QLIKE_HAR": ql["M1_HAR"].mean(), "ganho_vs_HAR_%": 100 * d.mean() / ql["M1_HAR"].mean(), "DM_t": t, "p_valor": p})
    # ---- Modelo B ----
    resB = []
    for (a, tipo, h), nome in escolha_B.items():
        ycol = alvo_col(a, tipo, h)
        g = grupos_B(a)
        if tipo == "novo":
            g = {k: [c for c in v if c != f"{a}_em_evento"] for k, v in g.items()}
        usar = sorted(set(g[nome] + g["B1"] + [ycol]))
        tr = dev.dropna(subset=usar); te = hold.dropna(subset=usar)
        ytr, yte = tr[ycol].to_numpy(float), te[ycol].to_numpy(float)
        if ytr.sum() < 20 or yte.sum() < 5:
            resB.append({"rede": a.upper(), "alvo": tipo, "h": h, "degrau_escolhido": nome, "n": len(te), "obs": "poucos eventos no holdout"}); continue
        p0 = np.full(len(te), ytr.mean())
        pe = logit_pred(logit_fit(tr[g[nome]].to_numpy(float), ytr), te[g[nome]].to_numpy(float))
        p1 = logit_pred(logit_fit(tr[g["B1"]].to_numpy(float), ytr), te[g["B1"]].to_numpy(float))
        d = (p1 - yte) ** 2 - (pe - yte) ** 2
        t, p = dm_hac(d, L=max(24, h) + h)
        resB.append({"rede": a.upper(), "alvo": tipo, "h": h, "degrau_escolhido": nome, "n": len(te), "prev_holdout": yte.mean(),
                     **{f"{k}_escolhido": v for k, v in metricas(yte, pe, p0).items()},
                     "BSS_B1": metricas(yte, p1, p0)["BSS"], "dBrier_vs_B1_%": 100 * d.mean() / np.mean((p1 - yte) ** 2),
                     "DM_t": t, "p_valor": p})
    # ---- Política operacional (esperar vs. agora) ----
    resP = []
    for a, nome in escolha_P.items():
        g = grupos_B(a)
        cols = g[nome]
        cst = pd.concat([dev, hold], ignore_index=True)[f"{a}_custo_usd"]
        base = pd.concat([dev, hold], ignore_index=True)
        base["delta"] = log_pos(cst.shift(-H_PRAZO)) - log_pos(cst.shift(-1))
        cref = cst.rolling(168, min_periods=84).median()
        tr = base.iloc[:len(dev)].dropna(subset=cols + ["delta"]); te = base.iloc[len(dev):].dropna(subset=cols + ["delta"])
        beta, _ = ols_fit(tr[cols].to_numpy(float), tr["delta"].to_numpy(float))
        tau = np.quantile(ols_pred(beta, tr[cols].to_numpy(float)), TAXA_PRIMARIA)
        pred = ols_pred(beta, te[cols].to_numpy(float))
        posi = te.index.to_numpy()
        F = np.column_stack([cst.shift(-k).to_numpy(float)[posi] for k in range(1, H_PRAZO + 1)])
        ref = cref.to_numpy(float)[posi]
        ok = np.isfinite(F).all(axis=1) & np.isfinite(ref) & (ref > 0)
        F, ref, pred = F[ok], ref[ok], pred[ok]
        espera = pred <= tau
        agora, prazo = F[:, 0], F[:, -1]
        modelo = np.where(espera, prazo, agora)
        idx = blocos_idx(len(F), B_BOOT)
        sv = np.array([(agora / ref)[i].mean() - (modelo / ref)[i].mean() for i in idx])
        resP.append({"rede": a.upper(), "degrau": nome, "n": len(F), "espera_%": 100 * espera.mean(),
                     "custo_rel_agora": (agora / ref).mean(), "custo_rel_modelo": (modelo / ref).mean(),
                     "custo_rel_prazo": (prazo / ref).mean(), "econ_vs_agora": (agora / ref).mean() - (modelo / ref).mean(),
                     "IC95_inf": np.percentile(sv, 2.5), "IC95_sup": np.percentile(sv, 97.5)})
    return pd.DataFrame(resA), pd.DataFrame(resB), pd.DataFrame(resP)

if ABRIR_HOLDOUT:
    if LOCK.exists() and not FORCAR_REABERTURA:
        raise RuntimeError(f"O holdout JÁ foi aberto ({LOCK.name}). Reabrir invalida o teste. Para forçar, defina FORCAR_REABERTURA = True e declare isso no trabalho.")
    resA_h, resB_h, resP_h = avaliar_holdout()
    resA_h["p_Holm"] = holm(resA_h["p_valor"].to_numpy())
    if "p_valor" in resB_h:
        pv = resB_h["p_valor"].fillna(1.0).to_numpy(); resB_h["p_Holm"] = holm(pv)
    resA_h.to_csv(PASTA_SAIDA / "holdout_modelo_A.csv", index=False)
    resB_h.to_csv(PASTA_SAIDA / "holdout_modelo_B.csv", index=False)
    resP_h.to_csv(PASTA_SAIDA / "holdout_politica.csv", index=False)
    with open(LOCK, "w", encoding="utf-8") as f:
        json.dump({"aberto_em": str(pd.Timestamp.now(tz="UTC")), "escolha_A": {f"{a}_{h}": m for (a, h), m in escolha_A.items()},
                   "escolha_B": {f"{a}_{t}_{h}": m for (a, t, h), m in escolha_B.items()},
                   "escolha_politica": {a: m for a, m in escolha_P.items()}, "taxa_espera": TAXA_PRIMARIA}, f, ensure_ascii=False, indent=2)
    print("HOLDOUT ABERTO (única vez). Modelo A:")
    display(resA_h.round(4))
    print("\nModelo B:")
    display(resB_h.round(4))
    print("\nPolítica operacional (esperar vs. agora):")
    display(resP_h.round(4))
else:
    print("Holdout permanece FECHADO (ABRIR_HOLDOUT = False).")
    print("Especificações já congeladas para a abertura:")
    print(" - Modelo A:", {f"{a.upper()}-{h}h": m for (a, h), m in escolha_A.items()})
    print(" - Modelo B:", {f"{a.upper()}-{t}-{h}h": m for (a, t, h), m in escolha_B.items()})
    print(" - Política:", {a.upper(): m for a, m in escolha_P.items()}, "| taxa de espera:", TAXA_PRIMARIA)

## 9. Saídas e síntese calculada

In [ ]:
corr_fng.to_csv(PASTA_SAIDA / "fng_vs_taxa_correcao.csv", index=False)
deriva.to_csv(PASTA_SAIDA / "deriva_eventos.csv", index=False)
tab_vol.to_csv(PASTA_SAIDA / "modeloA_qlike.csv", index=False)
inc_vol.to_csv(PASTA_SAIDA / "modeloA_incremental_vs_M3.csv", index=False)
met_B.to_csv(PASTA_SAIDA / "modeloB_metricas.csv", index=False)
inc_B.to_csv(PASTA_SAIDA / "modeloB_incremental.csv", index=False)
sim.to_csv(PASTA_SAIDA / "politica_operacional.csv", index=False)
tab_delta.to_csv(PASTA_SAIDA / "previsao_delta_custo.csv", index=False)
inc_delta.to_csv(PASTA_SAIDA / "previsao_delta_incremental.csv", index=False)
df_all[["hora", "particao"] + cols_novas].to_csv(PASTA_SAIDA / "features_novas_05.csv", index=False)

print("=== SÍNTESE CALCULADA ===")
# 1) Modelo A: algum degrau melhora robustamente o M3?
rob = inc_vol[(inc_vol["p_Holm"] < 0.05) & (inc_vol["ganho_QLIKE_%"] > 0)]
pio = inc_vol[(inc_vol["p_Holm"] < 0.05) & (inc_vol["ganho_QLIKE_%"] < 0)]
print(f"[Modelo A] variantes que melhoram o M3 de forma robusta (Holm<0,05): {len(rob)} de {len(inc_vol)}"
      + (": " + ", ".join(f"{r.rede}-{r.h}h {r['modelo vs M3']}" for _, r in rob.iterrows()) if len(rob) else ""))
print(f"[Modelo A] variantes que PIORAM o M3 de forma significativa: {len(pio)} de {len(inc_vol)}")

# 2) Modelo B: valor incremental do mercado e dos demais blocos (logística)
lg = inc_B[inc_B["modelo"] == "logit"]
for deg in ("B1 -> B2", "B2 -> B3", "B3 -> B4", "B1 -> B4"):
    s = lg[lg["degrau"] == deg]
    if len(s):
        print(f"[Modelo B] {deg}: melhora robusta em {int((s['veredito'] == 'melhora robusta').sum())}/{len(s)} testes | "
              f"piora significativa em {int((s['veredito'] == 'PIORA significativa').sum())}/{len(s)} | dBrier mediano = {s['dBrier_%'].median():.2f}%")
s = lg[lg["degrau"] == "B0 -> B1"]
print(f"[Modelo B] estado da própria taxa (B0 -> B1): dBrier mediano = {s['dBrier_%'].median():.1f}%")

# 3) Operacional
r2 = tab_delta[tab_delta["degrau"] == "B4"].set_index("rede")["R2_oos_vs_média"]
print("[Operacional] R² fora da amostra de Δ = ln custo(t+6) − ln custo(t+1) com todos os blocos (B4): "
      + ", ".join(f"{k} = {v:.3f}" for k, v in r2.items()))
if len(sim):
    prim = sim[sim["espera_alvo"] == TAXA_PRIMARIA]
    for _, r in prim[prim.apply(lambda x: x["degrau"] == escolha_P[x["rede"].lower()], axis=1)].iterrows():
        print(f"[Operacional] {r['rede']} ({r['degrau']}, espera em {r['espera_%']:.1f}% das horas): economia vs. 'agora' = {r['econ_vs_agora']:+.4f} "
              f"(custo relativo) IC95 {r['IC95_agora']} | vs. 'prazo' = {r['econ_vs_prazo']:+.4f} IC95 {r['IC95_prazo']}")
print("\nArquivos em:", PASTA_SAIDA)

## Próximos passos

1. **Ler as seções 5 a 7 antes de qualquer decisão.** Se o B2 não melhorar o B1, a conclusão honesta é que o mercado não agrega à previsão da taxa além do próprio estado da taxa.
2. **Sentimento textual:** sem GDELT, a hipótese central do trabalho não é testável. Opções com pouco volume de dados: (i) a API DOC do GDELT cobre ~3 meses, o que serve só como estudo exploratório em janela curta, nunca para o *walk-forward* de 2 anos; (ii) índices de volatilidade implícita (Deribit DVOL) e taxa de financiamento (Binance futuros) são séries horárias pequenas e gratuitas que funcionam como proxies de sentimento/estresse; (iii) manter o Fear & Greed e o fluxo de ordens como "sentimento disponível" e **declarar a limitação**.
3. **Abrir o *holdout* uma única vez** (seção 8), com a validação final do Modelo A e do Modelo B.
4. Redigir as conclusões com os números de `modeloA_*`, `modeloB_*`, `politica_operacional.csv` e `holdout_*.csv`.